In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from nice import NICE
import os

# Load the dataset
path = os.path.join("../dataset.csv")

dataset = pd.read_csv(path)

# Drop unnecessary columns
dataset = dataset.drop(['seqn', 'Marital'], axis='columns')

# Encode categorical variables and fill missing values
sex_mapping = {'Male': 0, 'Female': 1}
race_mapping = {'White': 0, 'Asian': 1, 'Black': 2, 'MexAmerican': 3, 'Hispanic': 4, 'Other': 5}
dataset['Sex'] = dataset['Sex'].replace(sex_mapping)
dataset['Race'] = dataset['Race'].replace(race_mapping)
dataset.iloc[:, 2] = dataset.iloc[:, 2].fillna(dataset.iloc[:, 2].mean())
dataset.iloc[:, 4] = dataset.iloc[:, 4].fillna(dataset.iloc[:, 4].mean())
dataset.iloc[:, 5] = dataset.iloc[:, 5].fillna(dataset.iloc[:, 5].mean())

# Split the dataset into features and target
X = dataset.drop('MetabolicSyndrome', axis=1)
y = dataset['MetabolicSyndrome']

# Create training and testing sets
outcome_0 = dataset[dataset['MetabolicSyndrome'] == 0]
outcome_1 = dataset[dataset['MetabolicSyndrome'] == 1]
test_size_each_class = 400
test_0 = outcome_0.sample(n=test_size_each_class, random_state=42)
test_1 = outcome_1.sample(n=test_size_each_class, random_state=42)
test_data = pd.concat([test_0, test_1])
train_data = dataset.drop(test_data.index)

x_train = train_data.drop('MetabolicSyndrome', axis=1).values
y_train = train_data['MetabolicSyndrome'].values
x_test = test_data.drop('MetabolicSyndrome', axis=1).values
y_test = test_data['MetabolicSyndrome'].values

# Train an XGBoost classifier
classifier_xgboost = XGBClassifier()
classifier_xgboost.fit(x_train, y_train)

# Define categorical and numerical features
cat_feat = ['Sex', 'Race']
num_feat = [
    'Age', 'Income', 'WaistCirc', 'BMI', 'Albuminuria', 'UrAlbCr',
    'UricAcid', 'BloodGlucose', 'HDL', 'Triglycerides'
]

# Get indices of categorical and numerical features
feature_indices = {name: idx for idx, name in enumerate(X.columns)}
cat_feat_indices = [feature_indices[feat] for feat in cat_feat]
num_feat_indices = [feature_indices[feat] for feat in num_feat]

# Initialize the NICE explainer
NICE_explainer = NICE(
    X_train=x_train,
    predict_fn=classifier_xgboost.predict_proba,
    y_train=y_train,
    cat_feat=cat_feat_indices,
    num_feat=num_feat_indices
)

# Function to generate counterfactual examples with predictions
def generate_counterfactuals_with_predictions(explainer, classifier, x_train, y_train):
    class_0_indices = np.where(y_train == 0)[0]
    class_1_indices = np.where(y_train == 1)[0]
    
    counterfactuals = []
    original_instances = []
    original_predictions = []
    counterfactual_predictions = []
    
    for i, idx in enumerate(class_0_indices):
        try:
            instance = x_train[idx].reshape(1, -1)
            cf_example = explainer.explain(instance)
            orig_pred = classifier.predict(instance)[0]
            cf_pred = classifier.predict(cf_example)[0]
            
            counterfactuals.append(cf_example)
            original_instances.append(instance)
            original_predictions.append(orig_pred)
            counterfactual_predictions.append(cf_pred)
        except Exception as e:
            print(f"Error processing instance {i}: {str(e)}")
            continue
    
    for i, idx in enumerate(class_1_indices):
        try:
            instance = x_train[idx].reshape(1, -1)
            cf_example = explainer.explain(instance)
            orig_pred = classifier.predict(instance)[0]
            cf_pred = classifier.predict(cf_example)[0]
            
            counterfactuals.append(cf_example)
            original_instances.append(instance)
            original_predictions.append(orig_pred)
            counterfactual_predictions.append(cf_pred)
        except Exception as e:
            print(f"Error processing instance {i}: {str(e)}")
            continue
    
    return original_instances, counterfactuals, original_predictions, counterfactual_predictions

# Generate counterfactual examples with predictions
original_instances, counterfactuals, original_predictions, counterfactual_predictions = \
    generate_counterfactuals_with_predictions(NICE_explainer, classifier_xgboost, x_train, y_train)

# Convert counterfactual examples to DataFrame
feature_names = X.columns
counterfactual_dfs = [pd.DataFrame(cf, columns=feature_names) for cf in counterfactuals]
original_dfs = [pd.DataFrame(orig, columns=feature_names) for orig in original_instances]

# Create pairs of original and counterfactual examples with predictions
pairs = []
for i, (orig, cf) in enumerate(zip(original_dfs, counterfactual_dfs)):
    orig['Outcome'] = original_predictions[i]
    cf['Outcome'] = counterfactual_predictions[i]
    pair = pd.concat([orig, cf], axis=0)
    pair.index = ['Original', 'Counterfactual']
    pairs.append(pair)

# Concatenate all pairs into a single DataFrame
result_df = pd.concat(pairs, axis=0)

# Export the result to a CSV file
result_df.to_csv('counterfactual_pairs_all_with_outcomes.csv', index=True)